# W4A4 target-precision QAT sweep

This run uses the fixed 45,000/5,000 train-validation split. The held-out CIFAR-10 test set is evaluated exactly once, after validation has selected a checkpoint from epochs that actually ran at W4A4.

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2

In [ ]:
from pathlib import Path

# Update these two dataset paths to the Kaggle dataset versions attached to this notebook.
BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/baseline/baseline.pt'
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'

!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/sweeps
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt
!sha256sum results/checkpoints/baseline.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
assert all((cifar_dir / name).is_file() for name in required), f'Missing CIFAR-10 files under {cifar_dir}'
print('Using fixed split: 45,000 train / 5,000 validation; official test is held out.')

In [ ]:
# Includes the projection-boundary and quantizer-math unit tests.
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/w4a4-correctness.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/w4a4-gpu.log

In [ ]:
RUN_NAME = 'w4a4-boundaryq-scalefix-seed6886'  # Unique: sweep records are immutable.
WEIGHT_BITS, ACTIVATION_BITS = 4, 4
EPOCHS = 12

assert (WEIGHT_BITS, ACTIVATION_BITS) == (4, 4)
print(f'Launching {RUN_NAME}: W{WEIGHT_BITS}A{ACTIVATION_BITS}; 8→6→4 transition, then target-only selection.')

In [ ]:
!set -o pipefail; python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "{DATA_DIR}" --device cuda --weight-bits {WEIGHT_BITS} --activation-bits {ACTIVATION_BITS} --epochs {EPOCHS} --run-name {RUN_NAME} 2>&1 | tee results/logs/{RUN_NAME}.log

In [ ]:
import csv
import json
import torch
from IPython.display import Image, display

run_dir = Path('experiments/sweeps') / RUN_NAME
checkpoint_path = Path('results/checkpoints') / f'qat-{RUN_NAME}-best-target.pt'
metrics = json.loads((run_dir / 'metrics.json').read_text())
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert checkpoint['quantization_state']['weight_bits'] == WEIGHT_BITS
assert checkpoint['quantization_state']['activation_bits'] == ACTIVATION_BITS
assert checkpoint['epoch'] >= 3, 'A W4A4 target checkpoint cannot come from the transition epochs.'
print('Selected target checkpoint:', checkpoint_path)
print('Epoch:', checkpoint['epoch'])
print('Best target validation accuracy:', metrics['best_target_validation_accuracy'])
diagnostics_path = Path(metrics['activation_diagnostics'])
assert diagnostics_path.is_file(), f'Missing activation diagnostics: {diagnostics_path}'
print('Activation diagnostics:', diagnostics_path)
with diagnostics_path.open(newline='') as handle:
    rows = list(csv.DictReader(handle))
target_rows = [row for row in rows if row['phase'] == 'validation' and int(row['bits']) == ACTIVATION_BITS]
worst = sorted(target_rows, key=lambda row: float(row['saturation_percent']), reverse=True)[:10]
print('Most saturated target-precision validation boundaries:')
for row in worst:
    print(f"{row['quantizer']}: {row['saturation_percent']}% saturated; scale={row['scale']}; clip=[{row['clip_min']}, {row['clip_max']}]")
display(Image(filename=str(run_dir / 'history.png')))

In [ ]:
# Run once. This result is not used to choose epochs, precision, or hyperparameters.
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/qat-{RUN_NAME}-best-target.pt --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/{RUN_NAME}-held-out-test.log

In [ ]:
import hashlib
import tarfile
from IPython.display import FileLink

paths = [run_dir, Path('results/logs') / f'{RUN_NAME}.log', Path('results/logs') / f'{RUN_NAME}-held-out-test.log', checkpoint_path, Path('results/checkpoints') / f'qat-{RUN_NAME}-latest.pt']
assert all(path.exists() for path in paths)
for path in paths:
    if path.is_file():
        print(f'{hashlib.sha256(path.read_bytes()).hexdigest()}  {path}')
archive = Path(f'{RUN_NAME}-artifacts.tgz')
with tarfile.open(archive, 'w:gz') as tar:
    for path in paths:
        tar.add(path, arcname=str(path))
print(f'Created {archive} ({archive.stat().st_size:,} bytes)')
FileLink(str(archive))